# Estudio de segmentación de tejidos en DFUTissue (Colab / GPU)

Corre en Colab con **GPU T4** (`Entorno de ejecución -> Cambiar tipo -> T4 GPU`).
Tres experimentos, cada uno guarda sus resultados directo a tu Google Drive:

1. **Eficiencia de anotación (T3+T4, bien hecho).** ¿Cuántas máscaras densas hacen falta,
   y *aporta algo la anotación débil* o es solo el modelo aprendiendo de N muestras?
   Compara dos curvas: `mixto` (N máscaras + resto etiquetas) vs `solo_supervisado` (solo N).
2. **Superar el baseline de Italia** (Fibrina 0.333 / Granulación 0.786 / Callo 0.515) con
   FPN+MobileNetV2 + receta fuerte (aumentación fuerte + pérdida Tversky + sobre-muestreo de fibrina).
3. **Comparación de modelos** con la receta fuerte: FPN+MobileNetV2 (embebido) vs Unet++ / Unet+EfficientNet / SegFormer (cotas).

Todo el código vive en el repo público; este notebook solo lo orquesta.

## 1 · Preparación (clonar repo, instalar, bajar datos, montar Drive)

In [ ]:
import torch, os
print('CUDA disponible:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Activa la GPU: Entorno de ejecucion -> Cambiar tipo -> T4 GPU'

%cd /content
![ -d Co_MIL_PPS ] || git clone --depth 1 https://github.com/AdrianbeltranFC/Co_MIL_PPS
%pip -q install segmentation-models-pytorch==0.5.0

# Datos publicos (DFUTissue) -> se descargan frescos, no vienen en el repo
!cd Co_MIL_PPS && python CO-MIL/segmentacion/descargar_datos.py

from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/comil_resultados_dfutissue'
os.makedirs(DRIVE_OUT, exist_ok=True)

# Que todo lo que los scripts escriben en Pesos_Entrenados/ caiga directo en Drive
# (sobrevive a una desconexion de Colab)
!rm -rf Co_MIL_PPS/Pesos_Entrenados && ln -s {DRIVE_OUT} Co_MIL_PPS/Pesos_Entrenados
print('resultados ->', DRIVE_OUT)

In [ ]:
# Verificacion rapida del dataset
!cd Co_MIL_PPS && python CO-MIL/segmentacion/dataset_seg.py

## 2 · Experimento 1 — Eficiencia de anotación (¿aporta la anotación débil?)

N &isin; {5, 10, 20, 40, 78} &times; 3 semillas &times; 2 modos. La **brecha entre las dos curvas**
(`mixto` &minus; `solo_supervisado`) es la respuesta. ~2&ndash;3 h en T4.
Guarda incrementalmente en `Drive/comil_resultados_dfutissue/eficiencia_<fecha>/`.

In [ ]:
!cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_eficiencia.py \
    --mascaras 5,10,20,40,78 --semillas 42,1,7 --modos mixto,solo_supervisado \
    --epocas 150 --paciencia 25

## 3 · Experimento 2 — Superar el baseline de Italia (receta fuerte, 78 máscaras densas)

FPN + MobileNetV2, supervisión densa completa, con: aumentación fuerte + pérdida **Tversky**
(castiga más los falsos negativos, buena para la fibrina) + **sobre-muestreo** de las imágenes
con fibrina. 3 semillas para tener media &plusmn; desviación. ~30&ndash;45 min en T4.

In [ ]:
for s in (42, 1, 7):
    !cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_seg.py \
        --arch FPN --encoder mobilenet_v2 --perdida tversky --aug_fuerte --sobremuestreo \
        --epocas 300 --paciencia 45 --semilla {s}

## 4 · Experimento 3 — Comparación de modelos (misma receta fuerte)

El de MobileNetV2 (Exp. 2) es el **titular embebido**. Estos son cotas superiores:
Unet++ con ResNet34 (estilo Italia), Unet con EfficientNet-B0, y SegFormer-B0
(familia del paper original de DFUTissue). ~1 h en T4.

In [ ]:
modelos = [
    ('UnetPlusPlus', 'resnet34'),
    ('Unet', 'efficientnet-b0'),
    ('Segformer', 'mit_b0'),
]
for arch, enc in modelos:
    !cd Co_MIL_PPS && python -u CO-MIL/segmentacion/entrenar_seg.py \
        --arch {arch} --encoder {enc} --perdida tversky --aug_fuerte --sobremuestreo \
        --epocas 300 --paciencia 45 --semilla 42

## 5 · Recolectar todo y hacer la figura resumen

In [ ]:
import json, glob, numpy as np, matplotlib.pyplot as plt

ITALIA = {'Fibrina': 0.333, 'Granulación': 0.786, 'Callo': 0.515}

# --- resumen de los experimentos 2 y 3 (segmentacion densa) ---
filas = []
for md in sorted(glob.glob(f'{DRIVE_OUT}/seg_exp_*/metadata.json')):
    d = json.load(open(md))
    t = d['test']['dice_por_clase']
    filas.append((d['arquitectura'], d.get('semilla'), t['Fibrina'], t['Granulación'],
                  t['Callo'], d['test']['dice_medio_tejidos']))
print(f"{'modelo':<34}{'sem':>4}{'Fib':>8}{'Gra':>8}{'Cal':>8}{'Media':>8}")
for f in filas:
    print(f"{f[0]:<34}{str(f[1]):>4}{f[2]:>8.3f}{f[3]:>8.3f}{f[4]:>8.3f}{f[5]:>8.3f}")
print(f"\n{'ITALIA (ResUnet+Unet++)':<34}{'':>4}{ITALIA['Fibrina']:>8.3f}"
      f"{ITALIA['Granulación']:>8.3f}{ITALIA['Callo']:>8.3f}{np.mean(list(ITALIA.values())):>8.3f}")

# --- la curva de eficiencia (experimento 1) ---
efs = sorted(glob.glob(f'{DRIVE_OUT}/eficiencia_*/resultados.json'))
if efs:
    ag = json.load(open(efs[-1]))['agregado_por_n']
    fig, ax = plt.subplots(figsize=(8, 5))
    for modo, col, ls in [('mixto', '#1C6B63', '-'), ('solo_supervisado', '#B4791A', '--')]:
        sub = sorted([a for a in ag if a['modo'] == modo], key=lambda a: a['n_mascaras'])
        if not sub: continue
        ns = [a['n_mascaras'] for a in sub]
        m = np.array([a['dice_medio_tejidos_media'] for a in sub])
        s = np.array([a['dice_medio_tejidos_std'] for a in sub])
        lbl = 'N máscaras + etiquetas débiles' if modo == 'mixto' else 'solo N máscaras'
        ax.plot(ns, m, 'o', ls=ls, color=col, lw=2, label=lbl)
        ax.fill_between(ns, m - s, m + s, color=col, alpha=0.15)
    ax.axhline(np.mean(list(ITALIA.values())), color='#888', ls=':', label='baseline Italia (media)')
    ax.set_xlabel('Nº de máscaras densas (de 78)'); ax.set_ylabel('Dice medio en tejidos')
    ax.set_title('Eficiencia de anotación — DFUTissue (3 semillas, ±1σ)')
    ax.set_ylim(0, 1); ax.grid(alpha=.3); ax.legend()
    fig.tight_layout(); fig.savefig(f'{DRIVE_OUT}/resumen_eficiencia.png', dpi=140)
    plt.show()

print('\nTodo guardado en', DRIVE_OUT, '- descarga esa carpeta y avisa.')